# 00 SP - Conversão CSV → Parquet

Converte o arquivo `35_SP.csv` (3.8GB) do Censo 2022 para formato Parquet (Snappy).

**Execute este notebook uma única vez antes dos demais.**

- Origem : `data/censo 2022/35_SP.csv`
- Destino : `data/censo 2022/35_SP.parquet`

O Parquet ocupa ~70% menos espaço e é lido muito mais rápido pelo DuckDB.

In [ ]:
from pathlib import Path

CSV_PATH     = Path('../data/censo 2022/35_SP.csv')
PARQUET_PATH = Path('../data/censo 2022/35_SP.parquet')

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {CSV_PATH}')

import os
tamanho_gb = CSV_PATH.stat().st_size / 1e9
print(f'Arquivo CSV  : {CSV_PATH.name}')
print(f'Tamanho      : {tamanho_gb:.1f} GB')
print(f'Destino      : {PARQUET_PATH}')

In [ ]:
import duckdb, time

if PARQUET_PATH.exists():
    print(f'Parquet já existe ({PARQUET_PATH.stat().st_size / 1e9:.1f} GB). Pulando conversão.')
else:
    print('Convertendo... (pode levar alguns minutos para 3.8 GB)')
    con = duckdb.connect()
    t = time.time()
    con.execute(f"""
        COPY (
            SELECT * FROM read_csv('{CSV_PATH}', delim=';', header=true, auto_detect=true)
        )
        TO '{PARQUET_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    elapsed = time.time() - t
    tamanho_parquet = PARQUET_PATH.stat().st_size / 1e9
    print(f'\nConcluído em {elapsed:.0f}s')
    print(f'CSV    : {tamanho_gb:.1f} GB')
    print(f'Parquet: {tamanho_parquet:.1f} GB  ({(1 - tamanho_parquet/tamanho_gb)*100:.0f}% menor)')

In [ ]:
# Validação: checar schema e contagem de linhas
con = duckdb.connect()
PARQUET = f"'{PARQUET_PATH}'"

total = con.execute(f"SELECT COUNT(*) FROM read_parquet({PARQUET})").fetchone()[0]
print(f'Total de linhas: {total:,}')
print()
con.execute(f"DESCRIBE SELECT * FROM read_parquet({PARQUET}) LIMIT 1").df()